# 01: Exploratory Data Analysis (EDA)
## Campus Placement Predictor

This notebook explores the dataset distribution, missing values, correlation structures, and demographic placement rates across gender and college tier.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.data.loader import DataLoader
from src.utils.config import load_config

# Configure visual styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11

### 1. Load Datasets (Unbiased vs Biased)

In [ ]:
loader = DataLoader()
df_unbiased = loader.load_synthetic_data(biased=False)
df_biased = loader.load_synthetic_data(biased=True)
print(f'Unbiased shape: {df_unbiased.shape}')
print(f'Biased shape:   {df_biased.shape}')
df_unbiased.head()

### 2. Missing Values and Schema Summary

In [ ]:
print('Missing values check:')
display(df_unbiased.isnull().sum())

print('\nSummary Statistics:')
display(df_unbiased.describe().round(2))

### 3. Class Balance: Placement & Salary Bands

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Placement outcome
df_unbiased['placed'].value_counts(normalize=True).plot.bar(
    ax=axes[0], color=['#2b5c8f', '#e07a5f'], edgecolor='black'
)
axes[0].set_title('Overall Placement Rate (Unbiased)')
axes[0].set_xticklabels(['Placed (1)', 'Not Placed (0)'], rotation=0)
axes[0].set_ylabel('Proportion')

# Salary bands
df_unbiased[df_unbiased['placed'] == 1]['salary_band'].value_counts().plot.bar(
    ax=axes[1], color='#3d405b', edgecolor='black'
)
axes[1].set_title('Compensation Bands for Placed Students')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

### 4. Technical Skill Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_unbiased['cgpa'], kde=True, ax=axes[0, 0], color='#2a9d8f')
axes[0, 0].set_title('CGPA Distribution')

sns.histplot(df_unbiased['dsa_problems_solved'], kde=True, ax=axes[0, 1], color='#e76f51')
axes[0, 1].set_title('DSA Problems Solved Distribution')

sns.histplot(df_unbiased['aptitude_score'], kde=True, ax=axes[1, 0], color='#457b9d')
axes[1, 0].set_title('Aptitude Score Distribution')

sns.histplot(df_unbiased['communication_score'], kde=True, ax=axes[1, 1], color='#1d3557')
axes[1, 1].set_title('Communication Score Distribution')

plt.tight_layout()
plt.show()

### 5. Correlation Heatmap

In [ ]:
numeric_cols = df_unbiased.select_dtypes(include=[np.number]).columns
corr = df_unbiased[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='vlag', annot=True, fmt='.2f', vmin=-1, vmax=1, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6. Subgroup Placement Rates: Gender and College Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender
gender_comp = pd.DataFrame({
    'Unbiased': df_unbiased.groupby('gender')['placed'].mean() * 100,
    'Biased': df_biased.groupby('gender')['placed'].mean() * 100
})
gender_comp.plot.bar(ax=axes[0], color=['#2b5c8f', '#e63946'], edgecolor='black')
axes[0].set_title('Placement Rate by Gender (%)')
axes[0].set_ylabel('Placement Rate (%)')
axes[0].set_xticklabels(gender_comp.index, rotation=0)

# College Tier
tier_comp = pd.DataFrame({
    'Unbiased': df_unbiased.groupby('college_tier')['placed'].mean() * 100,
    'Biased': df_biased.groupby('college_tier')['placed'].mean() * 100
})
tier_comp.plot.bar(ax=axes[1], color=['#2b5c8f', '#e63946'], edgecolor='black')
axes[1].set_title('Placement Rate by College Tier (%)')
axes[1].set_ylabel('Placement Rate (%)')
axes[1].set_xticklabels([f'Tier {t}' for t in tier_comp.index], rotation=0)

plt.tight_layout()
plt.show()